# HOGENOM CCP Reconciliation with GPUREC

This notebook loads the AleRax-style HOGENOM CCP family manifest, evaluates it with GPUREC, and saves reconciliation summaries. The default run uses genewise rates from AleRax checkpoint files when they are available.

The main output is a per-family table with GPUREC likelihoods and root-placement posteriors for explicit `fp64` and `fp32` runs. A later cell computes and saves the full `Pi` matrix for one selected family.

In [ ]:
from __future__ import annotations

import json
import math
import os
import sys
import time
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
try:
    from tqdm.auto import tqdm
except ImportError:
    tqdm = None

REPO = Path.cwd()
if not (REPO / "gpurec").exists():
    REPO = Path("/home/enzo/Documents/git/gpurec/gpurec")
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from gpurec import GeneReconModel
from gpurec.api.autograd import _extract_parameters
from gpurec.core.forward import Pi_wave_forward
from gpurec.core.likelihood import E_fixed_point, compute_log_likelihood_root_rows
from gpurec.core.model import GeneDataset, parse_alerax_family_file

cu13_lib = Path("/home/enzo/miniforge3/lib/python3.12/site-packages/nvidia/cu13/lib")
if torch.cuda.is_available() and torch.version.cuda == "13.0" and cu13_lib.exists():
    ld_library_path = os.environ.get("LD_LIBRARY_PATH", "")
    if str(cu13_lib) not in ld_library_path:
        print("CUDA 13 NVRTC libraries are not on LD_LIBRARY_PATH.")
        print("If the first GPU cell fails with libnvrtc-builtins.so.13.0, restart Jupyter with:")
        print(f"LD_LIBRARY_PATH={cu13_lib}:$LD_LIBRARY_PATH jupyter lab")

print("repo", REPO)
print("torch", torch.__version__, "cuda", torch.cuda.is_available(), torch.version.cuda)

## Configuration

For a smoke test, keep `MAX_FAMILIES` small. Set it to `None` for all families with checkpoint rates. `RUN_CONFIGS` makes the `fp64` and `fp32` evaluations explicit. `FIXED_ITERS_E` and `FIXED_ITERS_PI` are the GPUREC evaluator settings; AleRax's current CLI has one shared `--dtl-iterations` knob, but GPUREC lets you separate them here.

In [ ]:
HOGENOM_DIR = REPO / "tests" / "data" / "HOGENOM" / "hogenom"
FAMILIES_FILE = HOGENOM_DIR / "hogenom_families.local.txt"
ALERAX_OUTPUT = HOGENOM_DIR / "output_alerax_corrected"
ALERAX_CHECKPOINT_DIR = ALERAX_OUTPUT / "checkpoint"
ALERAX_PER_FAM_LIKELIHOODS = ALERAX_OUTPUT / "per_fam_likelihoods.txt"

inferred_species_tree = ALERAX_OUTPUT / "species_trees" / "inferred_species_tree.newick"
SPECIES_TREE = inferred_species_tree if inferred_species_tree.exists() else HOGENOM_DIR / "hogenom_S.tree"

OUTPUT_ROOT = HOGENOM_DIR / "output_gpurec_ccp_reconciliation"
PREPROCESS_CACHE_DIR = OUTPUT_ROOT / "preprocess_cache"

DEVICE = "cuda"
MODE = "genewise"  # "genewise", "global", or "specieswise"

@dataclass(frozen=True)
class RunConfig:
    label: str
    dtype: torch.dtype

    @property
    def output_dir(self) -> Path:
        return OUTPUT_ROOT / self.label


RUN_CONFIGS = [
    RunConfig(label="fp64", dtype=torch.float64),
    RunConfig(label="fp32", dtype=torch.float32),
]

START_FAMILY = 0
MAX_FAMILIES = None  # Set to None for all checkpointed HOGENOM families.
CHUNK_SIZE = 100
EVALUATION_STRATEGY = "chunked"  # "resident" builds one model; "chunked" rebuilds per CHUNK_SIZE.
FALLBACK_TO_CHUNKED_ON_OOM = True
VERBOSE_STATUS = True
SHOW_PROGRESS_BARS = True
CAPTURE_CONVERGENCE_TRACE = True
REPORT_TRITON_COMPILATION = True
MAX_TRITON_COMPILE_MESSAGES = 12

USE_ALERAX_CHECKPOINT_RATES = True
DEFAULT_RATES = (0.1, 0.1, 0.1)

FIXED_ITERS_E = 16
FIXED_ITERS_PI = 16
NEUMANN_TERMS = 6
MAX_WAVE_SIZE = 32768
USE_PRUNING = True

if DEVICE == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("CUDA was requested but torch.cuda.is_available() is false")

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
PREPROCESS_CACHE_DIR.mkdir(parents=True, exist_ok=True)
for run in RUN_CONFIGS:
    run.output_dir.mkdir(parents=True, exist_ok=True)
print("families", FAMILIES_FILE)
print("species_tree", SPECIES_TREE)
print("output_root", OUTPUT_ROOT)
print("runs", [(run.label, str(run.dtype).replace("torch.", ""), run.output_dir) for run in RUN_CONFIGS])

In [ ]:
def read_alerax_reference_likelihoods(path: Path) -> dict[str, float]:
    if not path.exists():
        return {}
    out: dict[str, float] = {}
    for raw in path.read_text().splitlines():
        parts = raw.split()
        if len(parts) >= 2:
            out[parts[0]] = float(parts[1])
    return out


def read_checkpoint_rates(path: Path) -> tuple[float, float, float]:
    lines = path.read_text().splitlines()
    if len(lines) < 2:
        raise ValueError(f"checkpoint file has no rate line: {path}")
    values = [float(x) for x in lines[1].split()]
    if len(values) < 3:
        raise ValueError(f"checkpoint rate line has fewer than 3 values: {path}")
    return values[0], values[1], values[2]


def posterior_from_log2_rows(log_rows: torch.Tensor) -> torch.Tensor:
    shifted = log_rows - log_rows.max(dim=1, keepdim=True).values
    probs = torch.exp2(shifted)
    return probs / probs.sum(dim=1, keepdim=True)


all_family_names, all_tree_paths, all_leaf_maps = parse_alerax_family_file(FAMILIES_FILE)
tree_paths_by_name = dict(zip(all_family_names, all_tree_paths))
leaf_maps_by_name = dict(zip(all_family_names, all_leaf_maps))
family_rank_by_name = {name: i for i, name in enumerate(all_family_names)}

alerax_loglik_by_name = read_alerax_reference_likelihoods(ALERAX_PER_FAM_LIKELIHOODS)
rates_by_name: dict[str, tuple[float, float, float]] = {}
if ALERAX_CHECKPOINT_DIR.exists():
    for path in sorted(ALERAX_CHECKPOINT_DIR.glob("*.txt")):
        if path.stem not in family_rank_by_name:
            continue
        rates_by_name[path.stem] = read_checkpoint_rates(path)

available_names = list(all_family_names)
if USE_ALERAX_CHECKPOINT_RATES:
    available_names = [name for name in available_names if name in rates_by_name]

selected_names = available_names[START_FAMILY:]
if MAX_FAMILIES is not None:
    selected_names = selected_names[:MAX_FAMILIES]

print("families in manifest", len(all_family_names))
print("families with AleRax checkpoint rates", len(rates_by_name))
print("families with AleRax reference likelihoods", len(alerax_loglik_by_name))
print("selected families", len(selected_names))
selected_names[:5]

In [ ]:
def dtype_name(dtype: torch.dtype) -> str:
    return str(dtype).replace("torch.", "")


def status(message: str, run: RunConfig | None = None) -> None:
    if not VERBOSE_STATUS:
        return
    run_prefix = "" if run is None else f"[{run.label}]"
    print(f"[{time.strftime('%H:%M:%S')}]{run_prefix} {message}", flush=True)


def progress_enabled() -> bool:
    return bool(SHOW_PROGRESS_BARS and tqdm is not None)


class TritonCompileReporter:
    def __init__(self, run: RunConfig):
        self.run = run
        self.enabled = bool(REPORT_TRITON_COMPILATION)
        self.available = False
        self.specialization_misses = 0
        self.disk_cache_hits = 0
        self.actual_compiles = 0
        self._knobs = None
        self._old_cache_hook = None
        self._old_post_compile_hook = None
        self._old_listener = None

    def __enter__(self):
        if not self.enabled:
            return self
        try:
            from triton import knobs
        except Exception as exc:
            status(f"Triton compile hooks unavailable: {exc}", self.run)
            return self
        self._knobs = knobs
        self._old_cache_hook = knobs.runtime.jit_cache_hook
        self._old_post_compile_hook = knobs.runtime.jit_post_compile_hook
        self._old_listener = knobs.compilation.listener
        knobs.runtime.jit_cache_hook = self._jit_cache_hook
        knobs.runtime.jit_post_compile_hook = self._jit_post_compile_hook
        knobs.compilation.listener = self._compilation_listener
        self.available = True
        return self

    def __exit__(self, exc_type, exc, tb):
        if self._knobs is not None:
            self._knobs.runtime.jit_cache_hook = self._old_cache_hook
            self._knobs.runtime.jit_post_compile_hook = self._old_post_compile_hook
            self._knobs.compilation.listener = self._old_listener
        return False

    def _kernel_label(self, hook_kwargs: dict) -> str:
        label = str(hook_kwargs.get("repr") or "Triton kernel")
        return label.split("(", 1)[0]

    def _maybe_status(self, message: str, count: int) -> None:
        if count <= MAX_TRITON_COMPILE_MESSAGES:
            status(message, self.run)
        elif count == MAX_TRITON_COMPILE_MESSAGES + 1:
            status("additional Triton compile/cache messages suppressed", self.run)

    def _jit_cache_hook(self, **kwargs):
        old_result = None
        if self._old_cache_hook is not None:
            old_result = self._old_cache_hook(**kwargs)
        if old_result:
            return old_result
        self.specialization_misses += 1
        label = self._kernel_label(kwargs)
        self._maybe_status(f"Triton JIT cache miss: resolving {label}", self.specialization_misses)
        return None

    def _jit_post_compile_hook(self, **kwargs):
        if self._old_post_compile_hook is not None:
            self._old_post_compile_hook(**kwargs)

    def _compilation_listener(self, **kwargs):
        if self._old_listener is not None:
            self._old_listener(**kwargs)
        if kwargs.get("cache_hit"):
            self.disk_cache_hits += 1
            return
        self.actual_compiles += 1
        src = kwargs.get("src")
        label = getattr(src, "name", "Triton kernel")
        self._maybe_status(f"compiled Triton kernel: {label}", self.actual_compiles)

    def summary(self) -> str:
        if not self.enabled:
            return "Triton compilation reporting disabled"
        if not self.available:
            return "Triton compilation reporting unavailable"
        if self.specialization_misses == 0:
            return "Pi forward used in-process cached Triton kernels; no JIT work"
        return (
            f"Triton JIT summary: specializations={self.specialization_misses}, "
            f"compiled={self.actual_compiles}, disk_cache_hits={self.disk_cache_hits}"
        )


def chunked(iterable: list[str], size: int):
    for start in range(0, len(iterable), size):
        yield start, iterable[start:start + size]


def build_dataset(run: RunConfig, family_names: list[str]) -> GeneDataset:
    return GeneDataset(
        species_tree_path=str(SPECIES_TREE),
        gene_tree_paths=[tree_paths_by_name[name] for name in family_names],
        genewise=MODE == "genewise",
        specieswise=MODE == "specieswise",
        dtype=run.dtype,
        device=torch.device(DEVICE),
        preprocess_cache_dir=PREPROCESS_CACHE_DIR,
        family_names=family_names,
        leaf_species_maps=[leaf_maps_by_name[name] for name in family_names],
    )


def initial_theta(run: RunConfig, dataset: GeneDataset, family_names: list[str]) -> torch.Tensor:
    device = torch.device(DEVICE)
    if MODE == "genewise":
        rates = [rates_by_name.get(name, DEFAULT_RATES) for name in family_names]
        return torch.log2(torch.tensor(rates, dtype=run.dtype, device=device))
    base = torch.log2(torch.tensor(DEFAULT_RATES, dtype=run.dtype, device=device))
    if MODE == "specieswise":
        return base.unsqueeze(0).expand(int(dataset.S), -1).clone()
    return base


def build_model_for_names(run: RunConfig, family_names: list[str]) -> GeneReconModel:
    status(f"building dataset/model for {len(family_names)} families", run)
    dataset = build_dataset(run, family_names)
    status(f"building wave layout for {len(family_names)} families", run)
    model = GeneReconModel(
        dataset=dataset,
        mode=MODE,
        theta_init=initial_theta(run, dataset, family_names),
        fixed_iters_E=FIXED_ITERS_E,
        fixed_iters_Pi=FIXED_ITERS_PI,
        neumann_terms=NEUMANN_TERMS,
        max_wave_size=MAX_WAVE_SIZE,
        use_pruning=USE_PRUNING,
    )
    static = model.static
    status(
        f"model ready: dtype={dtype_name(run.dtype)} families={len(family_names)} "
        f"species={static.species_helpers['S']} clades={static.wave_layout['C']} "
        f"waves={len(static.wave_layout['wave_metas'])}",
        run,
    )
    return model


def make_e_progress(e_bar):
    def e_progress(iteration: int, total: int, max_diff, converged: bool) -> None:
        if e_bar is None:
            return
        e_bar.update(1)
        postfix = {"iter": f"{iteration}/{total}"}
        if max_diff is not None:
            postfix["maxdiff"] = f"{max_diff:.2e}"
        if converged:
            postfix["converged"] = True
        e_bar.set_postfix(postfix)
    return e_progress


def make_pi_progress(pi_bar):
    def pi_progress(event: str, wave_index, total_waves: int, local_iter, fixed_iters: int, meta) -> None:
        if pi_bar is None:
            return
        if event == "wave_start" and wave_index is not None:
            width = int(meta["W"]) if meta is not None else 0
            pi_bar.set_description(f"Pi wave {wave_index + 1}/{total_waves} W={width}")
        elif event == "dts_start" and wave_index is not None:
            pi_bar.set_postfix(stage="DTS", wave=wave_index + 1)
        elif event == "pi_iter" and wave_index is not None:
            pi_bar.update(1)
            pi_bar.set_postfix(wave=wave_index + 1, iter=local_iter)
    return pi_progress


def run_e_fixed_point(run: RunConfig, model: GeneReconModel, params: tuple[torch.Tensor, ...]) -> dict:
    static = model.static
    log_pS, log_pD, log_pL, max_transfer_vec = params
    e_max_iters = static.fixed_iters_E if static.fixed_iters_E is not None else static.max_iters_E
    e_tolerance = -1.0 if static.fixed_iters_E is not None else static.tol_E
    status(f"starting E fixed point: iters={e_max_iters} shape={tuple(max_transfer_vec.shape)}", run)
    e_bar = tqdm(total=e_max_iters, desc=f"{run.label} E fixed point", leave=False) if progress_enabled() else None
    try:
        return E_fixed_point(
            species_helpers=static.species_helpers,
            log_pS=log_pS,
            log_pD=log_pD,
            log_pL=log_pL,
            max_transfer_mat=max_transfer_vec,
            max_iters=e_max_iters,
            tolerance=e_tolerance,
            warm_start_E=None,
            dtype=static.dtype,
            device=static.device,
            ancestors_T=static.ancestors_T,
            progress_callback=make_e_progress(e_bar),
            trace_logsumexp=CAPTURE_CONVERGENCE_TRACE,
        )
    finally:
        if e_bar is not None:
            e_bar.close()


def run_pi_forward(run: RunConfig, model: GeneReconModel, e_out: dict, params: tuple[torch.Tensor, ...]) -> dict:
    static = model.static
    log_pS, log_pD, _log_pL, max_transfer_vec = params
    wave_count = len(static.wave_layout["wave_metas"])
    status(f"starting Pi forward: waves={wave_count} pi_iters_per_wave={static.fixed_iters_Pi}", run)
    total = wave_count * static.fixed_iters_Pi
    pi_bar = tqdm(total=total, desc=f"{run.label} Pi waves", leave=False) if progress_enabled() else None
    compile_reporter = TritonCompileReporter(run)
    try:
        with compile_reporter:
            pi_out = Pi_wave_forward(
                wave_layout=static.wave_layout,
                species_helpers=static.species_helpers,
                E=e_out["E"],
                Ebar=e_out["E_bar"],
                E_s1=e_out["E_s1"],
                E_s2=e_out["E_s2"],
                log_pS=log_pS,
                log_pD=log_pD,
                max_transfer_mat=max_transfer_vec,
                device=static.device,
                dtype=static.dtype,
                fixed_iters=static.fixed_iters_Pi,
                return_original=False,
                return_root_rows=True,
                family_idx=static.wave_layout.get("family_idx") if static.genewise else None,
                progress_callback=make_pi_progress(pi_bar),
                trace_root_logsumexp=CAPTURE_CONVERGENCE_TRACE,
            )
    finally:
        if pi_bar is not None:
            pi_bar.close()
    status(compile_reporter.summary(), run)
    return pi_out


@torch.no_grad()
def evaluate_root_reconciliation(run: RunConfig, model: GeneReconModel) -> tuple[torch.Tensor, torch.Tensor, dict[str, torch.Tensor | None]]:
    status("extracting D/L/T probabilities", run)
    params = _extract_parameters(model.theta.detach(), model.static)
    e_out = run_e_fixed_point(run, model, params)
    status(f"E fixed point done: iterations={e_out['iterations']}", run)
    pi_out = run_pi_forward(run, model, e_out, params)
    status("Pi forward done; computing root likelihood and root posterior", run)
    root_log2 = pi_out["Pi_root_rows"]
    nll_bits = compute_log_likelihood_root_rows(root_log2, e_out["E"])
    root_probs = posterior_from_log2_rows(root_log2)
    traces = {
        "E_logsumexp2": e_out["E_logsumexp_trace"],
        "Pi_root_logsumexp2": pi_out["root_logsumexp_trace"],
    }
    status("likelihood and posterior done", run)
    return nll_bits.detach(), root_probs.detach(), traces

## Run Reconciliation

Each configured run builds one resident model for all selected families. That avoids rebuilding preprocessing/layout state for every chunk. If the resident run exceeds GPU memory, the cell can fall back to chunked evaluation for that run.

In [ ]:
LN2 = math.log(2.0)


def clear_cuda_cache() -> None:
    if DEVICE == "cuda":
        torch.cuda.synchronize()
        torch.cuda.empty_cache()


def new_buffers() -> dict[str, list[dict[str, object]]]:
    return {"rows": [], "top_rows": [], "trace_rows": []}


def record_batch(
    run: RunConfig,
    buffers: dict[str, list[dict[str, object]]],
    start: int,
    batch_index: int,
    batch_names: list[str],
    model: GeneReconModel,
    nll_bits: torch.Tensor,
    root_probs: torch.Tensor,
) -> None:
    species_names = model.static.species_helpers["names"]
    top_prob, top_idx = torch.topk(root_probs, k=min(5, root_probs.shape[1]), dim=1)
    nll_bits_cpu = nll_bits.cpu().numpy()
    top_prob_cpu = top_prob.cpu().numpy()
    top_idx_cpu = top_idx.cpu().numpy()

    for local_i, name in enumerate(batch_names):
        rates = rates_by_name.get(name, DEFAULT_RATES)
        gpurec_loglik_nats = -float(nll_bits_cpu[local_i]) * LN2
        ref = alerax_loglik_by_name.get(name)
        buffers["rows"].append({
            "run": run.label,
            "dtype": dtype_name(run.dtype),
            "selection_rank": start + local_i,
            "family_file_rank": family_rank_by_name[name],
            "family": name,
            "D": rates[0],
            "L": rates[1],
            "T": rates[2],
            "gpurec_nll_bits": float(nll_bits_cpu[local_i]),
            "gpurec_loglik_nats": gpurec_loglik_nats,
            "alerax_loglik_nats": ref,
            "diff_nats": None if ref is None else gpurec_loglik_nats - float(ref),
            "top_root_species": species_names[int(top_idx_cpu[local_i, 0])],
            "top_root_species_index": int(top_idx_cpu[local_i, 0]),
            "top_root_probability": float(top_prob_cpu[local_i, 0]),
            "batch": batch_index,
        })
        for rank in range(top_prob_cpu.shape[1]):
            sp_i = int(top_idx_cpu[local_i, rank])
            buffers["top_rows"].append({
                "run": run.label,
                "dtype": dtype_name(run.dtype),
                "family": name,
                "rank": rank + 1,
                "species_index": sp_i,
                "species": species_names[sp_i],
                "probability": float(top_prob_cpu[local_i, rank]),
            })


def trace_value(trace: torch.Tensor | None, iteration: int, family_index: int):
    if trace is None or iteration < 0 or iteration >= int(trace.shape[0]):
        return None
    row = trace[iteration]
    return float(row) if row.ndim == 0 else float(row[family_index])


def record_convergence_trace(
    run: RunConfig,
    buffers: dict[str, list[dict[str, object]]],
    batch_index: int,
    batch_names: list[str],
    traces: dict[str, torch.Tensor | None],
) -> None:
    if not CAPTURE_CONVERGENCE_TRACE:
        return
    e_trace = traces.get("E_logsumexp2")
    pi_trace = traces.get("Pi_root_logsumexp2")
    if e_trace is None and pi_trace is None:
        return
    e_cpu = None if e_trace is None else e_trace.detach().cpu()
    pi_cpu = None if pi_trace is None else pi_trace.detach().cpu()
    e_iters = 0 if e_cpu is None else int(e_cpu.shape[0])
    pi_iters = 0 if pi_cpu is None else int(pi_cpu.shape[0])
    total_iters = max(e_iters, pi_iters)

    for local_i, name in enumerate(batch_names):
        e_final = trace_value(e_cpu, e_iters - 1, local_i)
        pi_final = trace_value(pi_cpu, pi_iters - 1, local_i)
        for iteration in range(total_iters):
            e_value = trace_value(e_cpu, iteration, local_i)
            pi_value = trace_value(pi_cpu, iteration, local_i)
            e_delta = None if e_value is None or e_final is None else e_value - e_final
            pi_delta = None if pi_value is None or pi_final is None else pi_value - pi_final
            buffers["trace_rows"].append({
                "run": run.label,
                "dtype": dtype_name(run.dtype),
                "batch": batch_index,
                "family": name,
                "iteration": iteration + 1,
                "E_logsumexp2": e_value,
                "E_logsumexp2_final": e_final,
                "E_logsumexp2_delta_to_final": e_delta,
                "E_logsumexp2_abs_delta_to_final": None if e_delta is None else abs(e_delta),
                "Pi_root_logsumexp2": pi_value,
                "Pi_root_logsumexp2_final": pi_final,
                "Pi_root_logsumexp2_delta_to_final": pi_delta,
                "Pi_root_logsumexp2_abs_delta_to_final": None if pi_delta is None else abs(pi_delta),
            })


def evaluate_one_batch(run: RunConfig, buffers: dict[str, list[dict[str, object]]], start: int, batch_index: int, batch_names: list[str]) -> None:
    model = build_model_for_names(run, batch_names)
    nll_bits, root_probs, traces = evaluate_root_reconciliation(run, model)
    record_batch(run, buffers, start, batch_index, batch_names, model, nll_bits, root_probs)
    record_convergence_trace(run, buffers, batch_index, batch_names, traces)
    clear_cuda_cache()
    del model, nll_bits, root_probs, traces


def evaluate_chunked_run(run: RunConfig, buffers: dict[str, list[dict[str, object]]], t0: float) -> None:
    total_batches = math.ceil(len(selected_names) / CHUNK_SIZE)
    for batch_index, (start, batch_names) in enumerate(chunked(selected_names, CHUNK_SIZE)):
        evaluate_one_batch(run, buffers, start, batch_index, batch_names)
        elapsed = time.perf_counter() - t0
        print(f"{run.label} chunk {batch_index + 1}/{total_batches} families={len(buffers['rows'])}/{len(selected_names)} elapsed_s={elapsed:.1f}")


def evaluate_resident_run(run: RunConfig, buffers: dict[str, list[dict[str, object]]], t0: float) -> None:
    print(f"{run.label} resident evaluation dtype={dtype_name(run.dtype)} families={len(selected_names)}")
    evaluate_one_batch(run, buffers, 0, 0, selected_names)
    elapsed = time.perf_counter() - t0
    print(f"{run.label} resident done families={len(buffers['rows'])}/{len(selected_names)} elapsed_s={elapsed:.1f}")


def write_run_outputs(run: RunConfig, buffers: dict[str, list[dict[str, object]]], elapsed_s: float) -> dict[str, object]:
    family = pd.DataFrame(buffers["rows"])
    top_roots = pd.DataFrame(buffers["top_rows"])
    trace = pd.DataFrame(buffers["trace_rows"])
    family_path = run.output_dir / "gpurec_hogenom_ccp_family_root_summary.csv"
    top_path = run.output_dir / "gpurec_hogenom_ccp_root_top5.csv"
    trace_path = run.output_dir / "gpurec_hogenom_ccp_convergence_trace.csv"
    family.to_csv(family_path, index=False)
    top_roots.to_csv(top_path, index=False)
    if CAPTURE_CONVERGENCE_TRACE and not trace.empty:
        trace.to_csv(trace_path, index=False)

    summary = {
        "run": run.label,
        "families": int(len(family)),
        "elapsed_s": float(elapsed_s),
        "mode": MODE,
        "device": DEVICE,
        "dtype": dtype_name(run.dtype),
        "fixed_iters_E": FIXED_ITERS_E,
        "fixed_iters_Pi": FIXED_ITERS_PI,
        "evaluation_strategy": EVALUATION_STRATEGY,
        "chunk_size": CHUNK_SIZE if EVALUATION_STRATEGY == "chunked" else None,
        "capture_convergence_trace": CAPTURE_CONVERGENCE_TRACE,
        "species_tree": str(SPECIES_TREE),
        "families_file": str(FAMILIES_FILE),
        "output_dir": str(run.output_dir),
        "family_csv": str(family_path),
        "root_top5_csv": str(top_path),
        "total_gpurec_loglik_nats": float(family["gpurec_loglik_nats"].sum()),
    }
    if CAPTURE_CONVERGENCE_TRACE and not trace.empty:
        summary["convergence_trace_csv"] = str(trace_path)
    if family["alerax_loglik_nats"].notna().any():
        valid = family.dropna(subset=["alerax_loglik_nats", "diff_nats"])
        summary.update({
            "families_with_alerax_reference": int(len(valid)),
            "total_alerax_loglik_nats": float(valid["alerax_loglik_nats"].sum()),
            "total_diff_nats": float(valid["diff_nats"].sum()),
            "mean_abs_diff_nats": float(valid["diff_nats"].abs().mean()),
            "max_abs_diff_nats": float(valid["diff_nats"].abs().max()),
        })
    (run.output_dir / "gpurec_hogenom_ccp_summary.json").write_text(json.dumps(summary, indent=2) + "\n")
    return {"run": run, "family_df": family, "top_root_df": top_roots, "trace_df": trace, "summary": summary}


def run_reconciliation(run: RunConfig) -> dict[str, object]:
    buffers = new_buffers()
    t0 = time.perf_counter()
    if EVALUATION_STRATEGY == "resident":
        try:
            evaluate_resident_run(run, buffers, t0)
        except RuntimeError as exc:
            message = str(exc).lower()
            if not FALLBACK_TO_CHUNKED_ON_OOM or "out of memory" not in message:
                raise
            print(f"{run.label} resident evaluation ran out of GPU memory; falling back to chunked evaluation")
            buffers = new_buffers()
            clear_cuda_cache()
            evaluate_chunked_run(run, buffers, t0)
    elif EVALUATION_STRATEGY == "chunked":
        evaluate_chunked_run(run, buffers, t0)
    else:
        raise ValueError("EVALUATION_STRATEGY must be 'resident' or 'chunked'")
    return write_run_outputs(run, buffers, time.perf_counter() - t0)


run_results = {run.label: run_reconciliation(run) for run in RUN_CONFIGS}
family_df = pd.concat([result["family_df"] for result in run_results.values()], ignore_index=True)
top_root_df = pd.concat([result["top_root_df"] for result in run_results.values()], ignore_index=True)
trace_df = pd.concat([result["trace_df"] for result in run_results.values()], ignore_index=True)
summary_df = pd.DataFrame([result["summary"] for result in run_results.values()])
summary = {"runs": [result["summary"] for result in run_results.values()]}

family_df.to_csv(OUTPUT_ROOT / "gpurec_hogenom_ccp_all_runs_family_root_summary.csv", index=False)
top_root_df.to_csv(OUTPUT_ROOT / "gpurec_hogenom_ccp_all_runs_root_top5.csv", index=False)
if CAPTURE_CONVERGENCE_TRACE and not trace_df.empty:
    trace_df.to_csv(OUTPUT_ROOT / "gpurec_hogenom_ccp_all_runs_convergence_trace.csv", index=False)
summary_df.to_csv(OUTPUT_ROOT / "gpurec_hogenom_ccp_all_runs_summary.csv", index=False)
(OUTPUT_ROOT / "gpurec_hogenom_ccp_all_runs_summary.json").write_text(json.dumps(summary, indent=2) + "\n")
summary_df

In [ ]:
display(family_df.head(20))
display(top_root_df.head(20))
display(summary_df)
print("saved", OUTPUT_ROOT / "gpurec_hogenom_ccp_all_runs_family_root_summary.csv")
print("saved", OUTPUT_ROOT / "gpurec_hogenom_ccp_all_runs_root_top5.csv")
if CAPTURE_CONVERGENCE_TRACE and not trace_df.empty:
    display(trace_df.head(40))
    print("saved", OUTPUT_ROOT / "gpurec_hogenom_ccp_all_runs_convergence_trace.csv")
print("saved", OUTPUT_ROOT / "gpurec_hogenom_ccp_all_runs_summary.json")

## Plots

In [ ]:
def plot_family_summaries(family: pd.DataFrame, output_root: Path) -> None:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), constrained_layout=True)
    for run_label, part in family.groupby("run"):
        axes[0].hist(part["gpurec_nll_bits"], bins=60, alpha=0.55, label=run_label)
        axes[1].hist(part["top_root_probability"], bins=60, alpha=0.55, label=run_label)
    axes[0].set_title("GPUREC per-family NLL")
    axes[0].set_xlabel("NLL (bits)")
    axes[0].set_ylabel("Families")
    axes[1].set_title("Top root placement posterior")
    axes[1].set_xlabel("posterior probability")
    axes[1].set_ylabel("Families")
    for ax in axes:
        ax.legend(title="run")
    fig.savefig(output_root / "gpurec_hogenom_ccp_all_runs_reconciliation_summary.png", dpi=160)
    plt.show()


def plot_alerax_comparison(family: pd.DataFrame, output_root: Path) -> None:
    if not family["alerax_loglik_nats"].notna().any():
        return
    valid = family.dropna(subset=["alerax_loglik_nats", "diff_nats"])
    fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)
    for run_label, part in valid.groupby("run"):
        axes[0].scatter(part["alerax_loglik_nats"], part["gpurec_loglik_nats"], s=10, alpha=0.55, label=run_label)
        axes[1].hist(part["diff_nats"], bins=60, alpha=0.55, label=run_label)
    lo = min(valid["alerax_loglik_nats"].min(), valid["gpurec_loglik_nats"].min())
    hi = max(valid["alerax_loglik_nats"].max(), valid["gpurec_loglik_nats"].max())
    axes[0].plot([lo, hi], [lo, hi], color="black", linewidth=1)
    axes[0].set_title("GPUREC vs AleRax likelihood")
    axes[0].set_xlabel("AleRax log-likelihood (nats)")
    axes[0].set_ylabel("GPUREC log-likelihood (nats)")
    axes[1].axvline(0.0, color="black", linewidth=1)
    axes[1].set_title("GPUREC - AleRax")
    axes[1].set_xlabel("difference (nats)")
    axes[1].set_ylabel("Families")
    for ax in axes:
        ax.legend(title="run")
    fig.savefig(output_root / "gpurec_hogenom_ccp_all_runs_alerax_comparison.png", dpi=160)
    plt.show()


def set_log_if_positive(ax, values: pd.Series) -> None:
    if values.dropna().gt(0).any():
        ax.set_yscale("log")


def plot_convergence_against_final(trace: pd.DataFrame, output_root: Path) -> None:
    if trace.empty:
        return
    columns = [
        "E_logsumexp2",
        "E_logsumexp2_final",
        "E_logsumexp2_abs_delta_to_final",
        "Pi_root_logsumexp2",
        "Pi_root_logsumexp2_final",
        "Pi_root_logsumexp2_abs_delta_to_final",
    ]
    means = trace.groupby(["run", "iteration"])[columns].mean(numeric_only=True).reset_index()
    fig, axes = plt.subplots(2, 2, figsize=(13, 9), constrained_layout=True)
    for run_label, part in means.groupby("run"):
        axes[0, 0].plot(part["iteration"], part["E_logsumexp2"], marker="o", linewidth=1.4, label=run_label)
        axes[0, 0].plot(part["iteration"], part["E_logsumexp2_final"], linestyle="--", linewidth=1.0, alpha=0.7)
        axes[0, 1].plot(part["iteration"], part["Pi_root_logsumexp2"], marker="o", linewidth=1.4, label=run_label)
        axes[0, 1].plot(part["iteration"], part["Pi_root_logsumexp2_final"], linestyle="--", linewidth=1.0, alpha=0.7)
        axes[1, 0].plot(part["iteration"], part["E_logsumexp2_abs_delta_to_final"], marker="o", linewidth=1.4, label=run_label)
        axes[1, 1].plot(part["iteration"], part["Pi_root_logsumexp2_abs_delta_to_final"], marker="o", linewidth=1.4, label=run_label)
    axes[0, 0].set_title("E logsumexp2 vs final")
    axes[0, 0].set_ylabel("mean logsumexp2(E)")
    axes[0, 1].set_title("Pi root logsumexp2 vs final")
    axes[0, 1].set_ylabel("mean root logsumexp2(Pi)")
    axes[1, 0].set_title("E distance to final")
    axes[1, 0].set_ylabel("mean abs(delta)")
    axes[1, 1].set_title("Pi root distance to final")
    axes[1, 1].set_ylabel("mean abs(delta)")
    set_log_if_positive(axes[1, 0], means["E_logsumexp2_abs_delta_to_final"])
    set_log_if_positive(axes[1, 1], means["Pi_root_logsumexp2_abs_delta_to_final"])
    for ax in axes.ravel():
        ax.set_xlabel("iteration")
        ax.legend(title="run")
    fig.savefig(output_root / "gpurec_hogenom_ccp_all_runs_convergence_vs_final.png", dpi=160)
    plt.show()


plot_family_summaries(family_df, OUTPUT_ROOT)
plot_alerax_comparison(family_df, OUTPUT_ROOT)
if CAPTURE_CONVERGENCE_TRACE:
    plot_convergence_against_final(trace_df, OUTPUT_ROOT)

## Materialize a Full Pi Matrix for One Family

`Pi` is saved in log2 space with shape `[clades, species]`. The row order is the original clade order for that family. This is useful for inspecting more than the root posterior without materializing the full HOGENOM matrix.

In [ ]:
PI_RUN_LABEL = "fp64"
PI_FAMILY_NAME = selected_names[0]
PI_RUN = {run.label: run for run in RUN_CONFIGS}[PI_RUN_LABEL]

pi_model = build_model_for_names(PI_RUN, [PI_FAMILY_NAME])
with torch.no_grad():
    pi_log2 = pi_model.pi_matrix(original_order=True).detach().cpu().numpy()
species_names = np.array(pi_model.static.species_helpers["names"], dtype=object)
rates = rates_by_name.get(PI_FAMILY_NAME, DEFAULT_RATES)

pi_path = PI_RUN.output_dir / f"{PI_FAMILY_NAME}.gpurec_pi_log2.npz"
np.savez_compressed(
    pi_path,
    run=np.array([PI_RUN.label], dtype=object),
    family=np.array([PI_FAMILY_NAME], dtype=object),
    species_names=species_names,
    pi_log2=pi_log2,
    rates=np.array(rates, dtype=float),
)
print("Pi shape", pi_log2.shape)
print("saved", pi_path)

root_id = int(pi_model._dataset.families[0]["root_clade_id"])
root_probs = posterior_from_log2_rows(torch.from_numpy(pi_log2[root_id:root_id + 1])).numpy()[0]
top = np.argsort(root_probs)[::-1][:10]
pd.DataFrame({
    "rank": np.arange(1, len(top) + 1),
    "species_index": top,
    "species": species_names[top],
    "root_probability": root_probs[top],
})